# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets (`@id` and name)
print("Available record sets:")
record_sets = dataset.record_sets
for rset in record_sets:
    print(f"@id: {rset.id} | name: {getattr(rset, 'name', '(no name)')}")

# Optionally, show fields for each record set
all_fields = dict()
for rset in record_sets:
    print(f"\nRecord set: {rset.id}")
    fields = rset.fields
    all_fields[rset.id] = []
    for field in fields:
        all_fields[rset.id].append({'@id': field.id, 'name': getattr(field, 'name', '')})
        print(f"  Field @id: {field.id} | name: {getattr(field, 'name', '')}")

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame. Reference record sets and fields by their `@id`.

In [ ]:
# Prepare to load data from all record sets
import collections

# Create a list of record set @ids
record_set_ids = [rset.id for rset in record_sets]

dataframes = collections.OrderedDict()

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows from record set '{record_set_id}' (columns: {list(df.columns)})\n")
    else:
        print(f"No records found for record set '{record_set_id}'\n")

# Example: show columns and head for first available record set
if dataframes:
    first_rs = next(iter(dataframes))
    print(f"\nFirst few rows from record set '@id': {first_rs}")
    print(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In this demonstration, we select a numeric field if available, and perform EDA using the `@id`s.

In [ ]:
# --- EDA SETUP: Select a record set and numeric field by @id ---
import numpy as np
selected_record_set_id = None
numeric_field_id = None
group_field_id = None

# Try to auto-select a record set with at least one numeric column
for rs_id, df in dataframes.items():
    # Heuristic: Pick the first float/int column as 'numeric_field', and another as 'group_field' if possible
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_cols:
        selected_record_set_id = rs_id
        numeric_field_id = numeric_cols[0]
        # for grouping: pick any non-numeric, e.g. string column
        group_candidates = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        group_field_id = group_candidates[0] if group_candidates else None
        break

if selected_record_set_id is None or numeric_field_id is None:
    print("No suitable numeric field found for EDA in the available record sets.")
else:
    print(f"Selected record set @id: {selected_record_set_id}")
    print(f"Numeric field @id: {numeric_field_id}")
    if group_field_id:
        print(f"Group field @id: {group_field_id}")

    # --- Filtering example (threshold: 10, change if necessary) ---
    threshold = 10
    numeric_series = dataframes[selected_record_set_id][numeric_field_id]
    filtered_df = dataframes[selected_record_set_id][numeric_series > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # --- Normalization example ---
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # --- Grouping example ---
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution and relationships between fields in the dataset.

Here, we create a histogram of the selected numeric field, and if group field exists, a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(dataframes[selected_record_set_id][numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in dataframes[selected_record_set_id].columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[selected_record_set_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to explore and process the FAIR² dataset, referencing entities by their `@id`. We loaded the dataset schema, listed available record sets and fields, loaded tabular data for analysis, and performed sample exploratory data analysis and simple visualizations on identified numeric features. 

This workflow demonstrates how FAIR datasets can be programmatically inspected and analyzed using standardized schema references for repeatability and clarity.

For further exploration, consult the field `@id`s and schema documentation to select features for your specific research questions or analyses.